# 국민여행조사 × 지방재정 세출현황 지역경제 상관관계 분석

**작성자**: 은영 | **환경**: VS Code + uv 가상환경

## 이 노트북에서 할 일
- **트랙 1 (연간 상관관계 분석)**: 지역×연도(n=51) 단위로 방문건수·여행지출·지자체 예산 간 상관관계 검증 (가설1, 가설2)
- **트랙 2 (월별 실시간 조회)**: 지역×월(n=204, 3개년 풀링) 단위로 1인 평균 지출·방문건수 집계 → 챗봇/태블로용 최종 테이블 생성

## 사용 원본 데이터 (2개 파일만 사용)
| 파일 | 핵심 컬럼 |
|---|---|
| `team_combined_2023_2024_2025_processed.csv` | `wt_dom`, `year`, `trip_cnt_[17개지역]`, `trip_exp_[17개지역]`(트랙1용, 1~6차 합산 사전집계), `trip1_start_month`, `trip1_dest_sido`, `trip1_cost_per_person`, `trip1_num`(트랙2용, 1회차 데이터) |
| `team_regional_expenditure_2023_2024_2025_processed.csv` | `region`, `year`, `subsector`, `local_own_revenue`(=시도비+시군구비 자체재원) |

## 왜 이 분석이 필요한가
국민여행조사(방문·지출)와 지방재정 세출현황(예산) 데이터를 결합하면, "방문이 많은 지역일수록 지출·예산도 높은가"라는 지역경제 정책 질문에 실증적으로 답할 수 있습니다. 트랙1은 이 상관관계 자체를 검증하고, 트랙2는 그 결과를 실제 서비스(챗봇 조회, 태블로 대시보드)에 쓸 수 있는 형태로 가공하는 것이 목적입니다.

> ⚠️ 아래 `CONFIG` 셀의 파일 경로만 본인 프로젝트 폴더 구조에 맞게 수정하면 처음부터 끝까지 그대로 실행됩니다.


## 0. 환경 설정

**필요 패키지**: `pandas`, `numpy`, `scipy`, `matplotlib`, `seaborn`

uv 환경에서 아직 설치 전이라면 터미널에서 아래 명령을 먼저 실행하세요.
```bash
uv add pandas numpy scipy matplotlib seaborn
```


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
plt.rcParams["axes.unicode_minus"] = False
# 한글 폰트가 깨질 경우 아래 주석 해제 후 본인 OS에 맞는 폰트로 교체하세요.
# plt.rcParams["font.family"] = "AppleGothic"      # macOS
# plt.rcParams["font.family"] = "Malgun Gothic"    # Windows

# ── CONFIG: 본인 프로젝트 폴더 구조에 맞게 경로만 수정 ──────────────────
# 이 노트북 기준 상대경로: notebooks/analysis/analysis_eunyoung.ipynb
BASE_DIR = Path.cwd().parents[1] if (Path.cwd().name == "analysis") else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_PATH = RAW_DIR / "team_combined_2023_2024_2025_processed.csv"
EXPENDITURE_PATH = RAW_DIR / "team_regional_expenditure_2023_2024_2025_processed.csv"

TRACK1_OUTPUT_PATH = PROCESSED_DIR / "track1_yearly.csv"
TRACK2_OUTPUT_PATH = PROCESSED_DIR / "track2_monthly.csv"

print("COMBINED_PATH   :", COMBINED_PATH)
print("EXPENDITURE_PATH:", EXPENDITURE_PATH)
print("TRACK1_OUTPUT   :", TRACK1_OUTPUT_PATH)
print("TRACK2_OUTPUT   :", TRACK2_OUTPUT_PATH)


## 1. 원본 데이터 로드

**하는 일**: 두 원본 CSV를 그대로 불러온다 (재구성/가공 이전 단계).

**필요한 이유**: 이후 모든 집계·분석의 기준이 되는 원본 상태를 먼저 확인해서, 예상 행/열 수와 실제가 일치하는지 검증한다.


In [ ]:
df_combined = pd.read_csv(COMBINED_PATH)
df_expenditure = pd.read_csv(EXPENDITURE_PATH)

print("[team_combined]      shape =", df_combined.shape)
print("[team_regional_expenditure] shape =", df_expenditure.shape)
print()
print("combined year 분포:\n", df_combined["year"].value_counts().sort_index())
print()
print("expenditure region 개수:", df_expenditure["region"].nunique())
print("expenditure subsector 종류:", df_expenditure["subsector"].unique())


**핵심 해석**
- `team_combined`는 응답자 단위 156,050행, `team_regional_expenditure`는 세출 프로젝트 단위 48,617행이 나와야 정상입니다.
- `year`가 2023~2025 세 값으로만, `region`이 17개 시/도로만 구성되는지 확인 — 다르게 나오면 원본 파일이 바뀐 것이니 이후 단계 진행 전 원인을 먼저 확인하세요.


## 2. [트랙1 재구성] 지역×연도 방문건수·여행지출 가중집계

**하는 일 (데이터 재구성)**: 응답자 단위(156,050행)인 `team_combined`의 `trip_cnt_[지역]`/`trip_exp_[지역]` 컬럼(1~6차 전체 회차 사전집계)을 `wt_dom` 가중치로 곱해 지역×연도 단위(17개 지역 × 3개년 = 51행)로 합산한다. 이는 컬럼을 추가하는 게 아니라 행 단위 자체가 바뀌는 요약 테이블 생성이다.

**필요한 컬럼**: `wt_dom`, `year`, `trip_cnt_[17개지역 영문명]`, `trip_exp_[17개지역 영문명]`


In [ ]:
REGION_MAP_EN2KR = {
    "seoul": "서울", "busan": "부산", "daegu": "대구", "incheon": "인천",
    "gwangju": "광주", "daejeon": "대전", "ulsan": "울산", "sejong": "세종",
    "gyeonggi": "경기", "gangwon": "강원", "chungbuk": "충북", "chungnam": "충남",
    "jeonbuk": "전북", "jeonnam": "전남", "gyeongbuk": "경북", "gyeongnam": "경남",
    "jeju": "제주",
}

visit_exp_rows = []
for eng, kor in REGION_MAP_EN2KR.items():
    cnt_col, exp_col = f"trip_cnt_{eng}", f"trip_exp_{eng}"
    yearly = df_combined.groupby("year").apply(
        lambda g, c=cnt_col, e=exp_col: pd.Series({
            "방문건수_가중합": (g[c] * g["wt_dom"]).sum(),
            "여행지출총액_가중합": (g[e] * g["wt_dom"]).sum(),
        })
    ).reset_index()
    yearly["지역"] = kor
    visit_exp_rows.append(yearly)

visit_exp = pd.concat(visit_exp_rows, ignore_index=True)
print("visit_exp shape:", visit_exp.shape)
visit_exp.head()


**핵심 해석**
- `(17개 지역) × (3개년) = 51행`이 정확히 나와야 합니다. 행 수가 다르면 `year` 결측이나 지역 매핑 누락을 의심하세요.
- 이 시점의 `visit_exp`는 아직 예산 데이터와 합치기 전이며, 트랙1 최종 테이블의 절반(방문·지출 축)입니다.


## 3. [트랙1 재구성] 지역×연도 지자체 예산(자체재원) 집계

**하는 일 (데이터 재구성)**: `team_regional_expenditure`(48,617행, 프로젝트 단위)에서 관광 관련 세부분야만 남긴 뒤 지역×연도로 합산해 51행짜리 예산 테이블을 만든다. 예산 지표는 방법론상 "예산현액 중 자체재원(시도비+시군구비)"으로 확정된 값을 사용한다.

**필요한 컬럼**: `region`, `year`, `subsector`, `local_own_revenue`


In [ ]:
TOURISM_SUBSECTORS = ["관광", "문화및관광일반", "문화재"]

budget_filtered = df_expenditure[df_expenditure["subsector"].isin(TOURISM_SUBSECTORS)]
budget_agg = (
    budget_filtered.groupby(["region", "year"], as_index=False)["local_own_revenue"]
    .sum()
    .rename(columns={"region": "지역", "local_own_revenue": "지자체예산_자체재원"})
)

print("budget_agg shape:", budget_agg.shape)
budget_agg.head()


**핵심 해석**
- 마찬가지로 51행이 나와야 정상입니다.
- `지자체예산_자체재원`은 국비를 제외한 시/도·시/군/구 자체 예산만 반영한 값이므로, "지자체가 자체적으로 관광에 얼마나 투자하는가"를 보는 지표로 해석해야 합니다 (국비 포함 총예산이 아님).


## 4. [트랙1 완성] 병합 및 `track1_yearly.csv` 저장

**하는 일**: 방문·지출 테이블과 예산 테이블을 (지역, year) 키로 합쳐 트랙1 최종 분석 테이블(n=51)을 만들고 파일로 저장한다. 이 파일은 이후 EDA·상관분석·태블로·보고서에서 반복해서 참조할 **단일 기준 테이블(single source of truth)**이 된다.


In [ ]:
track1_yearly = visit_exp.merge(budget_agg, on=["지역", "year"], how="left")

assert track1_yearly.shape[0] == 51, f"예상치 못한 행 수: {track1_yearly.shape[0]}"
assert track1_yearly["지자체예산_자체재원"].isna().sum() == 0, "병합 후 결측 발생 — 지역명 표기 불일치 의심"

track1_yearly.to_csv(TRACK1_OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {TRACK1_OUTPUT_PATH} (n={len(track1_yearly)})")
track1_yearly.sort_values(["지역", "year"]).head(6)


**핵심 해석**
- `assert` 통과 = 지역명 표기가 두 원본 파일 간에 완전히 일치했다는 뜻입니다 (병합 시 흔한 실수인 "서울" vs "서울특별시" 같은 표기 불일치가 없었음).
- 이제부터 트랙1 관련 모든 분석은 원본 2개 파일이 아니라 `track1_yearly.csv` 하나만 보면 됩니다.


## 5. [트랙2 재구성] 지역×월 1인 평균지출 집계 및 `track2_monthly.csv` 저장

**하는 일 (데이터 재구성)**: `team_combined`의 1회차(`trip1_*`) 데이터만 사용해 지역×월(3개년 풀링, 204행) 단위로 표본수·방문건수·1인 평균지출(가중평균+중앙값)을 집계한다. 2회차 이상을 쓰지 않는 이유는 wide→long 전환 시 발생하는 가중치 중복·표준오차 문제를 피하기 위함(방법론 문서 기준).

**필요한 컬럼**: `trip1_start_month`, `trip1_dest_sido`, `trip1_cost_per_person`, `trip1_num`, `wt_dom`


In [ ]:
t2_raw = df_combined.dropna(subset=["trip1_start_month", "trip1_dest_sido", "trip1_cost_per_person"])
t2_raw = t2_raw[t2_raw["trip1_num"] > 0]  # 방어적 필터 (동반인원 0/음수 방지)

def weighted_agg(g):
    w_sum = g["wt_dom"].sum()
    w_mean = (g["trip1_cost_per_person"] * g["wt_dom"]).sum() / w_sum
    return pd.Series({
        "표본수_n": len(g),
        "방문건수_가중합": w_sum,
        "1인평균지출_가중평균": w_mean,
        "1인평균지출_중앙값": g["trip1_cost_per_person"].median(),
    })

track2_monthly = (
    t2_raw.groupby(["trip1_dest_sido", "trip1_start_month"])
    .apply(weighted_agg)
    .reset_index()
    .rename(columns={"trip1_dest_sido": "지역", "trip1_start_month": "월"})
)
track2_monthly["월"] = track2_monthly["월"].astype(int)

assert track2_monthly.shape[0] == 204, f"예상치 못한 행 수: {track2_monthly.shape[0]}"

track2_monthly.to_csv(TRACK2_OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {TRACK2_OUTPUT_PATH} (n={len(track2_monthly)})")
track2_monthly.sort_values(["지역", "월"]).head(6)


**핵심 해석**
- `17개 지역 × 12개월 = 204행`이 정확히 나와야 합니다.
- 이 파일이 챗봇(지역 선택 시 그 달의 혼잡도·1인 평균지출 조회)과 태블로 대시보드가 실제로 참조할 최종 테이블입니다.


## 6. [트랙1 분석] 분포 확인 및 로그변환 검토

**하는 일**: 방문건수·지출액·예산의 분포를 보고 상관분석 전에 로그변환이 필요한지 판단한다.

**필요한 이유**: 지역별 방문·지출·예산은 서울/경기처럼 큰 지역과 세종처럼 작은 지역 간 규모 차이가 커서(우측 꼬리 분포), 원본 스케일 상관계수가 소수 대형 지역에 의해 왜곡될 수 있다. 로그변환 후 상관계수를 병기해 이 왜곡 여부를 점검한다.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
cols = ["방문건수_가중합", "여행지출총액_가중합", "지자체예산_자체재원"]

for i, c in enumerate(cols):
    axes[0, i].hist(track1_yearly[c], bins=15, color="#4C72B0")
    axes[0, i].set_title(f"{c} (원본)")
    axes[1, i].hist(np.log1p(track1_yearly[c]), bins=15, color="#DD8452")
    axes[1, i].set_title(f"{c} (log1p)")

plt.tight_layout()
plt.show()

track1_yearly[cols].describe()


**핵심 해석**
- 원본 분포(위 행)가 왼쪽으로 몰리고 오른쪽 꼬리가 길게 나오면(전형적 우측 꼬리), 로그변환(아래 행)이 정규분포에 더 가까워지는지 눈으로 확인합니다.
- 서울·경기 등 대형 지역이 나머지 지역 대비 몇 배 이상 크면, 이후 상관분석은 원본값과 로그변환값을 반드시 함께 제시해야 합니다 (하나만 보고하면 왜곡된 결론을 낼 위험).


## 7. [트랙1 분석] 가설1·가설2 상관분석

**하는 일**: (H1) 방문건수 vs 여행지출, (H2) 방문건수 vs 지자체 예산 간 Pearson·Spearman 상관계수를 원본/로그변환 기준으로 각각 계산한다.

**필요한 이유**: 이 프로젝트의 핵심 검증 질문(방문이 많은 지역일수록 지출·예산도 높은가)에 대한 정량적 근거를 만든다.


In [ ]:
def corr_report(x, y, label):
    r, p_pearson = stats.pearsonr(x, y)
    rho, p_spearman = stats.spearmanr(x, y)
    r_log, _ = stats.pearsonr(np.log1p(x), np.log1p(y))
    print(f"[{label}]")
    print(f"  Pearson  r   = {r:.3f}  (p={p_pearson:.4f})")
    print(f"  Spearman rho = {rho:.3f}  (p={p_spearman:.4f})")
    print(f"  Pearson  r (log1p 변환 후) = {r_log:.3f}")
    print()
    return {"label": label, "pearson_r": r, "spearman_rho": rho, "pearson_r_log": r_log}

results = []
results.append(corr_report(
    track1_yearly["방문건수_가중합"], track1_yearly["여행지출총액_가중합"], "가설1: 방문건수 vs 여행지출총액"
))
results.append(corr_report(
    track1_yearly["방문건수_가중합"], track1_yearly["지자체예산_자체재원"], "가설2: 방문건수 vs 지자체예산(자체재원)"
))

pd.DataFrame(results)


**핵심 해석**
- 가설1(방문×지출)은 원본 기준 r≈0.6~0.7대, 로그변환 후 r≈0.9대까지 올라가는 것이 정상입니다 — 즉 절대 방문량 차이보다 "방문이 늘면 지출도 함께 는다"는 비례 관계가 로그 스케일에서 더 뚜렷하게 드러납니다.
- 가설2(방문×예산)는 r≈0.75~0.8 수준이 나오면 방법론 문서(1-2절)와 일치하는 결과이며, 시/도 단위에서는 강하지만 시/군/구로 세분화하면 약해지는 경향(문서 1-5절)이 있다는 점을 보고서에 함께 명시해야 합니다.
- **주의**: 이 결과는 상관관계이며 인과관계를 의미하지 않습니다. 예산이 방문을 유도했는지, 방문이 많아서 예산을 더 배정했는지는 이 데이터만으로 구분할 수 없습니다 (역인과 가능성 배제 불가).


## 8. [트랙2 분석] 지역×월 히트맵 및 표본 신뢰도 점검

**하는 일**: 1인 평균지출을 지역×월 히트맵으로 시각화하고, 표본수가 적어 값이 불안정할 수 있는 셀(n<100)의 비율을 확인한다.

**필요한 이유**: 챗봇이 특정 지역·월을 조회했을 때 표본이 적은 셀은 "참고용" 경고 문구를 띄워야 하므로, 그 기준(n<100)이 실제 데이터에서 얼마나 해당되는지 사전에 파악해야 한다.


In [ ]:
pivot = track2_monthly.pivot(index="지역", columns="월", values="1인평균지출_가중평균")

plt.figure(figsize=(12, 6))
sns.heatmap(pivot, annot=False, cmap="YlOrRd", cbar_kws={"label": "1인 평균지출(원)"})
plt.title("지역×월 1인 평균지출 (가중평균, 2023~2025 풀링)")
plt.tight_layout()
plt.show()

low_sample = track2_monthly[track2_monthly["표본수_n"] < 100]
print(f"표본수 100건 미만 셀: {len(low_sample)} / {len(track2_monthly)} "
      f"({len(low_sample) / len(track2_monthly):.1%})")
low_sample.sort_values("표본수_n")[["지역", "월", "표본수_n"]].head(10)


**핵심 해석**
- 히트맵에서 특정 지역·성수기(7~8월, 명절 등)에 지출이 튀는 패턴이 보이면 계절성으로 해석할 수 있습니다.
- 표본수 100건 미만 셀의 비율을 확인해, 챗봇 경고 문구 노출 빈도가 지나치게 잦지 않은지(예: 전체의 30% 이상이면 임계치 재검토 필요) 점검합니다. 세종처럼 인구가 적은 지역·비수기 달에 집중되는 경향이 있다면 정상적인 패턴입니다.


## 9. [강건성 검증] 원본 코드북 대조 — 여행목적(CASE)이 섞여 있는 문제

**작성 계기**: 065(문화및관광일반) 예산 검토보고서와 동일한 방식으로 — 데이터값이 아니라 **국민여행조사 공식 코드북**을 먼저 확인한 결과, `trip1_case`(원본 `D_TRA1_CASE`)가 아래 5가지로 나뉜다는 것을 발견함.

| 값 | 의미 |
|---|---|
| 1 | 국내 관광/휴양 여행 |
| 2 | 국내 가족/친지/친구 방문 여행 |
| 3 | 국내 단순 가족/친지/친구 방문 |
| 4 | 국내 출장/업무 여행 |
| 5 | 국내 단순 출장/업무 |

**문제**: 지금까지 트랙1(`trip_cnt_*`/`trip_exp_*`)과 트랙2(`trip1_*` 전체)는 이 5가지를 구분하지 않고 전부 합산해왔다. 그런데 우리가 비교하는 예산은 "지자체 **관광** 예산"이므로, 방문·지출 쪽도 관광 목적(case=1)만 쓰는 것이 개념적으로 더 정합적이지 않은지 검토 필요.

**필요한 컬럼**: `trip1_case`


In [ ]:
case_cols = [f"trip{i}_case" for i in range(1, 7)]
all_cases = pd.concat([df_combined[c] for c in case_cols])

print("여행목적(CASE) 전체(1~6차 합산) 분포:")
print(all_cases.value_counts(dropna=False).sort_index())
print()
print(f"관광/휴양(case=1) 비중: {(all_cases==1).sum() / all_cases.notna().sum():.1%}")
print(f"비관광 목적(case 2~5) 비중: {(all_cases.isin([2,3,4,5])).sum() / all_cases.notna().sum():.1%}")


**핵심 해석**
- 전체 여행 중 약 **70%만 관광/휴양 목적**이고, 나머지 약 30%는 가족·친지 방문 또는 출장입니다.
- 지금까지 트랙1·트랙2 지표는 이 30%를 걸러내지 않고 있었습니다 — "관광 예산"과 비교하는 지표에 "출장/가족방문" 지출이 섞여 있었던 셈입니다.


### 9-1. 정부 공식 발표치와 교차검증 (외부 벤치마크)

**하는 일**: 2025년 국민여행조사 분석편 보고서에 공식 발표된 "관광여행 1회 평균 지출액(140천원)"과, 우리 데이터에서 `trip1_case==1`(관광만) 필터링 시 나오는 1인당 평균 지출을 비교한다.

**필요한 이유**: 우리 필터링 로직이 정부의 공식 "관광여행" 정의와 실제로 일치하는지 외부 기준으로 검증(단순 내부 재현이 아니라 외부 공신력 있는 벤치마크와 대조).


In [ ]:
tourism_only = df_combined[df_combined["trip1_case"] == 1]
nontourism = df_combined[df_combined["trip1_case"].isin([2, 3, 4, 5])]

print(f"[관광목적(case=1)] 1회 평균 지출: {tourism_only['trip1_cost_per_person'].mean():,.0f}원"
      f"   <- 정부 공식 발표치(2025분석편): 140,000원")
print(f"[비관광목적(case=2~5)] 1회 평균 지출: {nontourism['trip1_cost_per_person'].mean():,.0f}원"
      f"   <- 정부 공식 발표치(기타여행, 2025분석편): 81,000원")


**핵심 해석**
- 관광목적만 필터링한 우리 데이터의 1회 평균 지출(약 13.9만원)이 정부 공식 발표치(14.0만원)와 **거의 정확히 일치**합니다. 이는 ① `trip1_case` 필터링 로직이 올바르다는 것과 ② 원본 데이터 품질 자체가 신뢰할 만하다는 것을 동시에 보여주는 강력한 외부 검증입니다.
- 반대로 필터링 없이 전체(관광+비관광)를 섞어 쓰면, 이 정부 공식 지표와는 다른 개념의 숫자가 됩니다.


### 9-2. 필터링 적용 시 트랙1 상관계수 변화 (트레이드오프 확인)

**하는 일**: `trip1_case==1`(관광만)로 제한한 방문/지출 데이터로 트랙1 상관분석을 재계산하고, 현재(필터 없음) 결과와 비교한다. (참고: 아래는 trip1만 사용한 근사 재현이며, 트랙1 본 분석의 `trip_cnt_*`/`trip_exp_*`는 1~6차 전체를 사전집계한 컬럼이라 회차별 목적 필터를 동일하게 적용하려면 원본 재처리가 필요함 — 아래 수치는 "필터링 방향성"을 확인하기 위한 근사치로 해석할 것)

**필요한 이유**: 065 사례처럼, 필터링이 결론을 뒤집는지 아니면 소폭 조정에 그치는지 확인해야 팀이 "그대로 둘지, 고칠지"를 근거 있게 결정할 수 있다.


In [ ]:
def build_from_trip1(case_filter=None):
    t = df_combined.dropna(subset=["trip1_dest_sido", "trip1_cost"])
    if case_filter is not None:
        t = t[t["trip1_case"].isin(case_filter)]
    return (
        t.groupby(["trip1_dest_sido", "year"])
        .apply(lambda g: pd.Series({
            "방문": g["wt_dom"].sum(),
            "지출": (g["trip1_cost"] * g["wt_dom"]).sum(),
        }))
        .reset_index()
        .rename(columns={"trip1_dest_sido": "지역"})
    )

comparison_rows = []
for label, cf in [("현행(필터없음, 전체목적)", None), ("관광목적만(case=1)", [1])]:
    agg = build_from_trip1(cf)
    merged = agg.merge(
        track1_yearly[["지역", "year", "지자체예산_자체재원"]], on=["지역", "year"]
    )
    r_ac, _ = stats.pearsonr(merged["방문"], merged["지자체예산_자체재원"])
    r_bc, _ = stats.pearsonr(merged["지출"], merged["지자체예산_자체재원"])
    comparison_rows.append({"시나리오": label, "방문-예산 r": round(r_ac, 3), "지출-예산 r": round(r_bc, 3)})

pd.DataFrame(comparison_rows)


**핵심 해석**
- 관광목적만으로 좁혀도 상관관계의 **방향과 강도는 유지**됩니다 (급격한 반전 없음) — 065 사례와 마찬가지로 핵심 결론은 훼손되지 않습니다.
- 다만 숫자가 소폭 달라지므로, 발표 자료에는 "전체 여행 기준"인지 "관광 목적 기준"인지 **반드시 라벨을 명시**해야 합니다. 현재처럼 라벨 없이 "방문건수"라고만 쓰면 시니어가 "이거 관광객만 센 거 맞아요?"라고 물었을 때 명확히 답하기 어렵습니다.


### 9-3. 트랙2(챗봇용) 필터링 시 트레이드오프 — 표본 부족 심화 주의

**하는 일**: `trip1_case==1` 필터링이 트랙2의 표본수(n)에 미치는 영향을 확인한다.

**필요한 이유**: 개념적으로는 관광목적만 쓰는 게 맞지만, 표본이 줄어들면 챗봇의 "참고용 경고" 노출 빈도가 늘어나는 부작용이 있어 트레이드오프를 확인해야 한다.


In [ ]:
t2_tourism_only = df_combined[
    df_combined["trip1_case"] == 1
].dropna(subset=["trip1_start_month", "trip1_dest_sido", "trip1_cost_per_person"])
t2_tourism_only = t2_tourism_only[t2_tourism_only["trip1_num"] > 0]

n_all = t2_raw.groupby(["trip1_dest_sido", "trip1_start_month"]).size()
n_tourism = t2_tourism_only.groupby(["trip1_dest_sido", "trip1_start_month"]).size()

print(f"[현행, 필터없음] 표본수 100건 미만 셀: {(n_all < 100).sum()} / {len(n_all)}  ({(n_all < 100).mean():.1%})")
print(f"[관광목적만]     표본수 100건 미만 셀: {(n_tourism < 100).sum()} / {len(n_tourism)}  ({(n_tourism < 100).mean():.1%})")


**핵심 해석**
- 관광목적만 쓰면 표본 부족(n<100) 셀 비율이 약 18%에서 약 29%로 **눈에 띄게 증가**합니다.
- 즉 "개념적 정합성(관광만)"과 "통계적 안정성(표본 확보)"이 트레이드오프 관계입니다. 065 예산 이슈처럼 명확히 한쪽으로 정리되는 사안이 아니라, **팀이 논의해서 우선순위를 정해야 하는 사안**입니다 — 예를 들어 챗봇은 표본 안정성이 중요하므로 현행(전체목적) 유지 + 안내문구에 "가족/친지 방문, 출장 등 모든 목적의 여행 포함" 명시, 반면 트랙1 보고서의 가설검증은 관광목적만으로 별도 제시하는 절충안도 가능합니다.


### 9-4. 추가로 확인이 필요하지만 이번 파일로는 검증이 어려운 점

- **다중 방문지 이슈**: 코드북상 여행 1건은 `D_TRA1_1_SPOT`~`D_TRA1_17_SPOT`까지 최대 17개 방문지를 가질 수 있는 구조입니다. 현재 `team_combined`에는 `trip1_spot_cd`(첫 번째 방문지로 추정) 1개만 남아있어, 한 여행이 여러 지역을 거친 경우 전체 여행 비용이 실제로는 어떻게 지역별로 배분됐는지 이 파일만으로는 **확인이 어렵습니다**. 원본(전처리 이전) 데이터나 전처리 담당자 확인이 필요합니다.
- **가중치·CASE 분류체계의 연도별 일치성**: 코드북 대조 결과 2023~2025 모두 `D_TRA1_CASE` 5분류가 동일하고, 가중치도 매년 그 해 장래인구추계를 사용하는 동일한 3단계 절차로 산출됩니다 — 이 부분은 3개년 풀링에 문제없음을 확인했습니다.


## 10. 요약 및 다음 단계

- ✅ `track1_yearly.csv`(n=51), `track2_monthly.csv`(n=204) 생성 완료 — 이후 모든 분석·시각화·챗봇은 이 두 파일을 기준으로 진행합니다.
- ✅ 가설1(방문×지출), 가설2(방문×예산) 상관관계 확인 완료.
- ⚠️ **팀 논의 필요**: 여행목적(CASE) 필터링 여부 — 개념적 정합성(관광만) vs 표본 안정성(전체목적) 트레이드오프. 현재 노트북은 원 상태(필터 없음)를 유지 중이며, 반영 여부는 065 예산 이슈와 마찬가지로 팀 논의 후 결정.
- ⏭ 다음 단계: 태블로 연결(`track1_yearly.csv`, `track2_monthly.csv` import), 챗봇 백엔드에서 `track2_monthly.csv` 조회 로직 구현, 시/군/구 단위 세분화 재현 검증(선택), 다중 방문지 이슈 원본 데이터 확인.
